In [ ]:
import pathlib, os, sys, datetime

# Pastikan notebook dijalankan dari folder notebooks/
NOTEBOOK_DIR = pathlib.Path.cwd()
if not (NOTEBOOK_DIR / '..' / 'data').exists():
    raise RuntimeError("Jalankan notebook ini dari folder 'notebooks/' agar path ke data ditemukan.")

PROJECT_ROOT = (NOTEBOOK_DIR / '..').resolve()
print(f"Project root: {PROJECT_ROOT}")

DATA_DIR       = PROJECT_ROOT / 'data'
DATASET_DIR    = DATA_DIR / 'dataset'           # sumber mentah
CLEAN_DIR      = DATA_DIR / 'dataset_clean'     # hasil crop & enhance
MODEL_DIR      = PROJECT_ROOT / 'models'
OUTPUT_DIR     = PROJECT_ROOT / 'output'

# Konfigurasi training
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 16          # sesuaikan dengan VRAM (6 GB aman di 16)
EPOCHS_PHASE1 = 20
EPOCHS_PHASE2 = 10
LEARNING_RATE_HEAD = 0.001
LEARNING_RATE_FT = 0.00001
USE_MIXED_PRECISION = True

# Flag logging
VERBOSE = True


In [ ]:
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Folder dataset mentah tidak ditemukan: {DATASET_DIR}")
print(f"Dataset mentah ditemukan: {DATASET_DIR}")

# Buat folder output jika belum ada
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Sesi dimulai: {datetime.datetime.now()}")
with open(OUTPUT_DIR / 'training_log.txt', 'a') as f:
    f.write(f"Sesi dimulai: {datetime.datetime.now()}\n")


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print(f"GPU ditemukan: {gpus[0]}")
    except:
        pass
else:
    print("WARNING: Tidak ada GPU, training akan lambat.")

if gpus and USE_MIXED_PRECISION:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision diaktifkan (FP16).")
elif not gpus and USE_MIXED_PRECISION:
    print("WARNING: Mixed precision diminta tapi GPU tidak ditemukan. Melanjutkan tanpa mixed precision.")
    USE_MIXED_PRECISION = False


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from tqdm import tqdm

# Tambahkan path root ke sys.path agar bisa import dari src
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from src.preprocessing import find_medicine_package, crop_and_enhance

print("✅ Library dan modul preprocessing siap.")


In [ ]:
def prepare_clean_dataset():
    clean_asli_dir = CLEAN_DIR / 'asli'
    clean_palsu_dir = CLEAN_DIR / 'palsu'
    os.makedirs(clean_asli_dir, exist_ok=True)
    os.makedirs(clean_palsu_dir, exist_ok=True)

    total = 0
    failed = 0

    for folder, clean_dir in [('asli', clean_asli_dir), ('palsu', clean_palsu_dir)]:
        raw_dir = DATASET_DIR / folder
        print(f"MEMPROSES FOLDER {folder.upper()}")
        if not raw_dir.exists():
            print(f"Folder {raw_dir} tidak ditemukan, lewati.")
            continue

        for fname in tqdm(os.listdir(raw_dir)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                img = cv2.imread(str(raw_dir / fname))
                if img is None:
                    failed += 1
                    continue

                rect = find_medicine_package(img)
                if rect is None:
                    failed += 1

                enhanced = crop_and_enhance(img, rect, img_height=IMG_HEIGHT, img_width=IMG_WIDTH)
                cv2.imwrite(str(clean_dir / fname), enhanced)
                total += 1

    # Tulis statistik ke log
    with open(OUTPUT_DIR / 'training_log.txt', 'a') as f:
        f.write(f"{datetime.datetime.now()} - prepare_clean_dataset: total={total}, failed={failed}\n")

    print(f"Total diproses: {total}")
    print(f"Gagal deteksi (tapi tetap diproses): {failed}")
    return total, failed, []


In [ ]:
def train_model_on_clean_dataset():
    dataset_path = str(CLEAN_DIR)
    batch_size = BATCH_SIZE

    train_ds = tf.keras.utils.image_dataset_from_directory(
        dataset_path,
        validation_split=0.2,
        subset='training',
        seed=123,
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=batch_size,
        label_mode='binary',
        class_names=['asli', 'palsu']
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        dataset_path,
        validation_split=0.2,
        subset='validation',
        seed=123,
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=batch_size,
        label_mode='binary',
        class_names=['asli', 'palsu']
    )

    # Augmentasi ringan
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomBrightness(0.1),
        tf.keras.layers.RandomContrast(0.1),
    ])

    normalization_layer = tf.keras.layers.Rescaling(1./255)

    train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
    train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
    val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

    # Class weights
    count_asli = len(os.listdir(CLEAN_DIR / 'asli'))
    count_palsu = len(os.listdir(CLEAN_DIR / 'palsu'))
    total_samples = count_asli + count_palsu
    weight_asli = total_samples / (2 * count_asli) if count_asli > 0 else 1.0
    weight_palsu = total_samples / (2 * count_palsu) if count_palsu > 0 else 1.0
    class_weight = {0: weight_asli, 1: weight_palsu}
    print(f"Class weights: {class_weight}")

    # Build model
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE_HEAD)
    if USE_MIXED_PRECISION:
        optimizer = tf.keras.mixed_precision.LossScaleOptimizer(optimizer)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / 'panadol_v4_clean_best.h5'),
        monitor='val_loss',
        save_best_only=True
    )

    # Tahap 1: Train head only
    print("\n" + "="*60)
    print("TAHAP 1: TRAINING HEAD ONLY")
    print("="*60)
    history1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_PHASE1,
        class_weight=class_weight,
        callbacks=[early_stop, checkpoint]
    )

    # Tahap 2: Fine-tuning
    print("\n" + "="*60)
    print("TAHAP 2: FINE-TUNING")
    print("="*60)
    base_model.trainable = True
    for layer in base_model.layers[:100]:
        layer.trainable = False

    optimizer_ft = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE_FT)
    if USE_MIXED_PRECISION:
        optimizer_ft = tf.keras.mixed_precision.LossScaleOptimizer(optimizer_ft)

    model.compile(
        optimizer=optimizer_ft,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    history2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_PHASE2,
        class_weight=class_weight,
        callbacks=[early_stop, checkpoint]
    )

    # Gabungkan history
    full_history = {}
    for key in history1.history:
        full_history[key] = history1.history[key] + history2.history[key]

    # Plot
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(full_history['accuracy'], label='Train')
    plt.plot(full_history['val_accuracy'], label='Validasi')
    plt.legend(); plt.title('Akurasi')
    plt.subplot(1,2,2)
    plt.plot(full_history['loss'], label='Train')
    plt.plot(full_history['val_loss'], label='Validasi')
    plt.legend(); plt.title('Loss')
    plt.show()

    loss, acc = model.evaluate(val_ds)
    print(f"Validation Accuracy: {acc*100:.2f}%")

    # Simpan model
    model.save(str(MODEL_DIR / 'panadol_v4_clean.h5'))
    print(f"✅ Model disimpan: {MODEL_DIR / 'panadol_v4_clean.h5'}")

    # Konversi ke TFLite
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    with open(str(MODEL_DIR / 'panadol_v4_clean.tflite'), 'wb') as f:
        f.write(tflite_model)
    print(f"✅ Model TFLite disimpan: {MODEL_DIR / 'panadol_v4_clean.tflite'}")

    # Logging metrik akhir
    with open(OUTPUT_DIR / 'training_log.txt', 'a') as f:
        f.write(f"{datetime.datetime.now()} - Training selesai, val_accuracy={acc*100:.2f}%\n")

    return model, full_history


In [ ]:
# Load model dengan pengecekan eksistensi
model_path = str(MODEL_DIR / 'panadol_v4_clean.h5')
if not os.path.exists(model_path):
    print(f"❌ Model belum tersedia di {model_path}. Jalankan training terlebih dahulu atau salin model ke folder models/.")
    print("Fungsi inferensi tidak akan dimuat. Lanjutkan training atau periksa path.")
    verify_model = None
else:
    verify_model = tf.keras.models.load_model(model_path)
    print(f"✅ Model verifikasi dimuat dari {model_path}")

def predict_medicine(image_path, threshold=0.5, confidence_floor=0.80, white_ratio_min=0.03, min_text_blobs=3):
    if verify_model is None:
        return "ERROR", "Model tidak tersedia. Tidak dapat melakukan inferensi.", 0.0, None

    img = cv2.imread(image_path)
    if img is None:
        return "ERROR", "Gambar tidak ditemukan", 0.0, None

    original = img.copy()
    rect = find_medicine_package(img)
    if rect is None:
        rect = (0, 0, img.shape[1], img.shape[0])

    enhanced = crop_and_enhance(img, rect, img_height=IMG_HEIGHT, img_width=IMG_WIDTH)
    if enhanced is None:
        return "ERROR", "Gagal memproses gambar", 0.0, original

    img_rgb = cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB)
    img_array = np.expand_dims(img_rgb, axis=0) / 255.0
    pred = verify_model.predict(img_array, verbose=0)[0][0]

    if pred >= threshold:
        label = "❌ PALSU"
        confidence = pred
    else:
        label = "✅ ASLI"
        confidence = 1.0 - pred

    if confidence < confidence_floor:
        return "DITOLAK", "❌ Objek tidak dikenal / bukan obat", confidence, enhanced

    hsv = cv2.cvtColor(enhanced, cv2.COLOR_BGR2HSV)
    mask_white = cv2.inRange(hsv, (0, 0, 180), (180, 50, 255))
    white_ratio = np.sum(mask_white > 0) / mask_white.size

    gray = cv2.cvtColor(enhanced, cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    text_blobs = sum(1 for cnt in contours if 50 < cv2.contourArea(cnt) < 5000)

    has_white = white_ratio >= white_ratio_min
    has_text = text_blobs >= min_text_blobs

    if not (has_white or has_text):
        return "DITOLAK", "❌ Objek bukan obat / tidak ada ciri kemasan", confidence, enhanced

    return "TERVERIFIKASI", label, confidence, enhanced


def predict_and_display(image_path, threshold=0.5):
    if verify_model is None:
        print("Model tidak dimuat. Tidak dapat melakukan inferensi.")
        return

    status, label, confidence, enhanced = predict_medicine(image_path, threshold)

    original = cv2.imread(image_path)
    original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(original_rgb)
    plt.title("Gambar Asli")
    plt.axis('off')

    if enhanced is not None:
        enhanced_rgb = cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB)
        plt.subplot(1, 2, 2)
        plt.imshow(enhanced_rgb)
        plt.title("Hasil Crop & Enhance")
        plt.axis('off')

    plt.suptitle(f"{label}\nConfidence: {confidence*100:.2f}%", fontsize=14)
    plt.show()

    print("="*60)
    print(f"File    : {os.path.basename(image_path)}")
    print(f"Status  : {status}")
    print(f"Hasil   : {label}")
    print(f"Confidence: {confidence*100:.2f}%")
    print("="*60)

    return status, label, confidence


In [ ]:
print("="*70)
print("LANGKAH 1: PERSIAPAN DATASET BERSIH (CROP & ENHANCE)")
print("="*70)
total, failed, _ = prepare_clean_dataset()

print("\n" + "="*70)
print("LANGKAH 2: TRAINING MODEL VERIFIKASI")
print("="*70)
model, history = train_model_on_clean_dataset()

print("\n✅ Pelatihan selesai! Model disimpan di folder models/")


In [ ]:
end_time = datetime.datetime.now()
print(f"Sesi berakhir: {end_time}")
with open(OUTPUT_DIR / 'training_log.txt', 'a') as f:
    f.write(f"Sesi selesai: {end_time}\n")
